# Week 1 — Make the first two model calls

**Research task:** Ask a hosted model and a model running on this computer to score the same synthetic statement for generalized social trust.

**Python introduced:** notebook cells, strings, lists, dictionaries, indexing, SDK response objects, attributes, methods, integers, Booleans, `print(...)`, `type(...)`, `len(...)` and a first research record.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christopherbarrie/GenAI_Soc2026/blob/main/workbook/session01/session01_research_technology.ipynb)

Colab supports the OpenRouter route only. Local JupyterLab or VS Code is canonical because it can also reach Ollama.

In [ ]:
# Colab setup: clone the public repository when running in Colab.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo = SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists():
        setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)], check=True)
    setup_os.chdir(setup_repo / 'workbook' / 'session01')
print('Working folder:', SetupPath.cwd())

## Load the course settings and SDKs

**Input:** installed Python packages, `config/course_models.json`, and—if it is not already set—the hidden OpenRouter key. **Operations:** `import` makes an installed tool available; `Path.cwd()` gives Python the current folder; the `while` block walks upward until it finds the course configuration; `json.loads(...)` turns the file's JSON text into a dictionary; square brackets retrieve the two model names. **Output:** `HOSTED_MODEL` and `LOCAL_MODEL` are strings. `getpass(...)` accepts the key without echoing it. The folder-search code is supplied setup and is not assessed.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Store the research material and construct one shared message list

`text` and `instruction` are strings. `+` joins those strings with the labelled statement to create `prompt`; `\n` means start a new line. The braces create one dictionary with two named fields, and the brackets place that dictionary in a list called `messages`. Both clients will receive this same list. `type(...)` reports the kind of Python object without changing it.


In [ ]:
text = "Most people can be trusted."
instruction = "Score the statement for generalized social trust from 0 to 2. Return one integer."
prompt = instruction + "\n\nStatement: " + text
messages = [{"role": "user", "content": prompt}]

print("text:", text)
print("type(text):", type(text))
print("messages:", messages)
print("type(messages):", type(messages))

## Send the message list through OpenRouter and unpack its return

The call receives three named inputs: the hosted model string, the message list and `temperature=0`. `with ... as client` opens the SDK client and closes it after the indented call. The service returns a response object containing a list named `choices`; `[0]` selects the first choice. `.message` retrieves its message and `.content` retrieves the returned text. `.strip()` removes surrounding whitespace but does not change the score itself.


In [ ]:
with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
    hosted_response = client.chat.send(
        model=HOSTED_MODEL,
        messages=messages,
        temperature=0,
    )

hosted_choice = hosted_response.choices[0]
hosted_message = hosted_choice.message
hosted_answer = hosted_message.content.strip()
print("OpenRouter raw text:", hosted_answer)

## Send the same message list through Ollama and unpack its return

`ollama.chat(...)` sends the same model-facing message list to the Ollama server on this computer. The local model name and temperature are separate named inputs. Ollama's response shape differs: the message is available as `.message`, without a `choices` list. The output is again a string. A local route changes where inference occurs; it does not validate the score.


In [ ]:
local_response = ollama.chat(
    model=LOCAL_MODEL,
    messages=messages,
    options={"temperature": 0},
)
local_message = local_response.message
local_answer = local_message.content.strip()
print("Ollama raw text:", local_answer)

## Use the request to learn strings, lists and dictionaries

Both calls have now run. `text` is a string. `messages` is an ordered list whose first position is `0`. `messages[0]` retrieves a dictionary and `=` assigns it to `first_message`; the keys `role` and `content` retrieve named fields. Parentheses pass a value to a function, so `len(messages)` calls a built-in function and returns an integer. We call `print`, `type` and `len`, but do not define a function until Week 9. These operations inspect the request already stored in Python; they do not contact either model.


In [ ]:
first_message = messages[0]

print("text value:", text)
print("text type:", type(text))
print("messages value:", messages)
print("messages type:", type(messages))
print("number of messages:", len(messages))
print("first message:", first_message)
print("first message type:", type(first_message))
print("role field:", first_message["role"])
print("content field:", first_message["content"])

## Use the returns to learn objects, attributes and methods

The SDKs return objects defined by their packages. Dot notation retrieves an attribute on one of those objects; numeric brackets retrieve a position in a list; a quoted key retrieves a field from a dictionary. `.strip()` is a string method that returns a new string without surrounding whitespace. Look at the type on the left before interpreting the dot or brackets.


In [ ]:
hosted_choices = hosted_response.choices

print("hosted response type:", type(hosted_response))
print("hosted choices type:", type(hosted_choices))
print("first hosted choice type:", type(hosted_choice))
print("hosted message type:", type(hosted_message))
print("hosted answer type:", type(hosted_answer))

print("local response type:", type(local_response))
print("local message type:", type(local_message))
print("local answer type:", type(local_answer))

## Derive an integer and a Boolean from the returned strings

`len(...)` counts characters and returns an integer. `==` compares the two returned strings and produces a Boolean: `True` or `False`. Agreement is a property of these two returns; it is not evidence that either score is valid.


In [ ]:
hosted_character_count = len(hosted_answer)
local_character_count = len(local_answer)
routes_match = hosted_answer == local_answer

print("hosted character count:", hosted_character_count)
print("count type:", type(hosted_character_count))
print("routes returned identical text:", routes_match)
print("comparison type:", type(routes_match))

## Preserve the two runs as separate research records

Each pair of braces creates a dictionary. Text before a colon is a field name; the value after it is what the record stores. The brackets around both records create a list. `research_records[0]["route"]` first selects item 0 from that list and then the `route` field from the chosen dictionary. The two outputs are not averaged or declared correct.


In [ ]:
hosted_record = {
    "route": "openrouter",
    "model": HOSTED_MODEL,
    "input": text,
    "raw_output": hosted_answer,
}
local_record = {
    "route": "ollama",
    "model": LOCAL_MODEL,
    "input": text,
    "raw_output": local_answer,
}

research_records = [hosted_record, local_record]

print(hosted_record)
print(local_record)
print("record collection type:", type(research_records))
print("first recorded route:", research_records[0]["route"])

# ONE CHANGE: replace text with "You cannot be too careful in dealing with people."
# Then rerun every cell from the shared message list onward.

## Methodological check

The two scores are proposed measurements. Neither route establishes that the score matches the intended construct or a defensible human codebook.
## Completion recording

Run both routes on the original statement and on `You cannot be too careful in dealing with people.` In the narrated recording, explain every string, the message list, where each inference runs, each returned object and each research record. Finish by explaining why either score still needs validation.

Explain every input and output aloud. Never show the shared key.